In [ ]:
pip install requests pandas

In [1]:
import requests
import pandas as pd
import time
import os
from datetime import datetime, timedelta

GUARDIAN_KEY = "8c2854a1-b950-49b2-bb3b-7da69be998de"

NYT_KEY = "eHDXcgVPQGHqEpkqn1WDpSWpbIDGvZrsMFG2GVeMggHxvgrm"

In [2]:
TOPICS = [
    ("Politics", "election OR politics OR democracy OR harris OR trump"),
    ("Economy", "inflation OR market OR economy OR gdp OR tariff"),
    ("Conflict", "war OR ukraine OR gaza OR israel OR russia OR military")
]

In [4]:
def get_month_ranges():
    months = []

    target_dates = []

    # Add all months of 2024 (1-12)
    for m in range(1, 13):
        target_dates.append((2024, m))

    # Add Jan to June of 2025 (1-6)
    for m in range(1, 7):
        target_dates.append((2025, m))

    # Loop through them to create the start/end strings
    for year, month in target_dates:
        start_date = datetime(year, month, 1)

        # Calculate end of month
        if month == 12:
            # End of Dec is day before Jan 1st of next year
            end_date = datetime(year + 1, 1, 1) - timedelta(days=1)
        else:
            # End of month is day before start of next month
            end_date = datetime(year, month + 1, 1) - timedelta(days=1)

        months.append({
            "start_g": start_date.strftime("%Y-%m-%d"), # Guardian Format (YYYY-MM-DD)
            "end_g": end_date.strftime("%Y-%m-%d"),
            "start_n": start_date.strftime("%Y%m%d"),   # NYT Format (YYYYMMDD)
            "end_n": end_date.strftime("%Y%m%d")
        })

    return months

print("Setup Complete. Keys ready. Date range set: Jan 2024 -> June 2025")

Setup Complete. Keys ready. Date range set: Jan 2024 -> June 2025


In [ ]:
def fetch_guardian_month(topic, query, start, end):
    articles = []
    page = 1
    max_pages = 3 # Aiming for 150 articles per topic/month

    url = "https://content.guardianapis.com/search"
    section_map = {
        "Politics": "politics|us-news|australia-news",
        "Economy": "business|money",
        "Conflict": "world"
    }

    while page <= max_pages:
        params = {
            'q': query,
            'section': section_map.get(topic),
            'from-date': start,
            'to-date': end,
            'api-key': GUARDIAN_KEY,
            'page-size': 50,
            'page': page,
            'show-fields': 'headline,bodyText,wordcount,trailText',
            'order-by': 'relevance'
        }
        try:
            r = requests.get(url, params=params)
            if r.status_code != 200: break
            data = r.json()
            results = data.get('response', {}).get('results', [])
            if not results: break

            for item in results:
                fields = item.get('fields', {})
                articles.append({
                    'headline': item.get('webTitle'),
                    'date': item.get('webPublicationDate')[:10],
                    'section': item.get('sectionName'),
                    'type': topic,
                    'source': 'The Guardian',
                    'text_snippet': fields.get('trailText', ''),
                    'full_text': fields.get('bodyText', '')[:2000]
                })
            page += 1
            time.sleep(0.2)
        except: break
    return articles

In [ ]:
print("--- STARTING GUARDIAN COLLECTION ---")
all_guardian_data = []
time_slices = get_month_ranges()

for period in time_slices:
    print(f"Processing: {period['start_g']} ...")
    for topic, query in TOPICS:
        data = fetch_guardian_month(topic, query, period['start_g'], period['end_g'])
        all_guardian_data.extend(data)

--- STARTING GUARDIAN COLLECTION ---
Processing: 2024-01-01 ...
Processing: 2024-02-01 ...
Processing: 2024-03-01 ...
Processing: 2024-04-01 ...
Processing: 2024-05-01 ...
Processing: 2024-06-01 ...
Processing: 2024-07-01 ...
Processing: 2024-08-01 ...
Processing: 2024-09-01 ...
Processing: 2024-10-01 ...
Processing: 2024-11-01 ...
Processing: 2024-12-01 ...
Processing: 2025-01-01 ...
Processing: 2025-02-01 ...
Processing: 2025-03-01 ...
Processing: 2025-04-01 ...
Processing: 2025-05-01 ...
Processing: 2025-06-01 ...


In [ ]:
df_guardian = pd.DataFrame(all_guardian_data)
df_guardian

,headline,date,section,type,source,text_snippet,full_text
0,Harris attacks Trump over Roe v Wade at first ...,2024-01-23,US news,Politics,The Guardian,Vice-president appeared Tuesday in Virginia wi...,Joe Biden and Kamala Harris kicked off their f...
1,Biden re-election campaign to put emphasis on ...,2024-01-03,US news,Politics,The Guardian,Biden-Harris camp announces campaign plan that...,"Good morning. Ailing in opinion polls, Joe Bid..."
2,Biden attacks Trump as grave threat to democra...,2024-01-05,US news,Politics,The Guardian,"On eve of January 6 anniversary, US president ...",A day before the third anniversary of the Janu...
3,Sherrilyn Ifill on a Trump win: ‘We will cease...,2024-01-07,US news,Politics,The Guardian,The longtime civil rights lawyer on the 14th a...,The timing is right for a 14th amendment renai...
4,Icy battle for democracy in Iowa with Trump ex...,2024-01-15,US news,Politics,The Guardian,The first stage of the 2024 election takes pla...,A cold coming we had of it. Icy winds blow acr...
...,...,...,...,...,...,...,...
7771,Russia bombards Kyiv after Putin vows revenge ...,2025-06-06,World news,Conflict,The Guardian,Three emergency workers killed and 20 people w...,Russia launched an intense missile and drone b...
7772,Nato secretary general says Europe needs to ‘r...,2025-06-12,World news,Conflict,The Guardian,Mark Rutte says ‘we have to spend more’ as min...,"… and on that note, it’s a wrap! Nato secretar..."
7773,Donald Trump repeats call for Russia to be rea...,2025-06-16,World news,Conflict,The Guardian,US president said Ukraine war would not have h...,Donald Trump has displayed his disdain for the...
7774,Russian forces closing in on Sumy city three y...,2025-06-08,World news,Conflict,The Guardian,Independent monitors confirm Kremlin claims of...,Russian military units appear to be within 18 ...


In [ ]:
guardian_filename = "guardian_2024_full.csv"
df_guardian.to_csv(guardian_filename, index=False)
print(f"\nSaved to file: {guardian_filename}")


Saved to file: guardian_2024_full.csv


In [ ]:
# Save to CSV
filename = "guardian_raw_data.csv"
df_guardian.to_csv(filename, index=False)

print(f"Data saved to {filename}")

Data saved to guardian_raw_data.csv


In [ ]:
# --- NYT FETCH FUNCTION ---
def fetch_nyt_month(topic, query, start, end):
    articles = []
    page = 0
    max_pages = 15 # Aiming for 150 articles per topic/month

    url = "https://api.nytimes.com/svc/search/v2/articlesearch.json"

    while page < max_pages:
        params = {
            'q': query,
            'begin_date': start,
            'end_date': end,
            'api-key': NYT_KEY,
            'page': page,
            'sort': 'relevance'
        }
        try:
            r = requests.get(url, params=params)

            if r.status_code == 429:
                time.sleep(10)
                continue
            if r.status_code != 200: break

            data = r.json()
            docs = data.get('response', {}).get('docs', [])
            if not docs: break

            for doc in docs:
                section = doc.get('section_name', 'Unknown')
                if section in ['Sports', 'Movies', 'Arts', 'Fashion']: continue

                if not doc.get('lead_paragraph'):
                    doc['lead_paragraph'] = doc.get('abstract')[:200]
                articles.append({
                    'headline': doc.get('headline', {}).get('main'),
                    'date': doc.get('pub_date')[:10],
                    'section': section,
                    'type': topic,
                    'source': 'New York Times',
                    'text_snippet': doc.get('lead_paragraph'),
                    'full_text': doc.get('abstract')
                })
            page += 1
            time.sleep(6)
        except: break
    return articles

In [ ]:
# --- MAIN EXECUTION ---
print("--- STARTING NYT COLLECTION (This takes ~15 mins) ---")
all_nyt_data = [] # List to hold everything
time_slices = get_month_ranges()

for period in time_slices:
    print(f"Processing: {period['start_n']} ...")
    for topic, query in TOPICS:
        data = fetch_nyt_month(topic, query, period['start_n'], period['end_n'])
        all_nyt_data.extend(data)

--- STARTING NYT COLLECTION (This takes ~15 mins) ---
Processing: 20240101 ...
Processing: 20240201 ...
Processing: 20240301 ...
Processing: 20240401 ...
Processing: 20240501 ...
Processing: 20240601 ...
Processing: 20240701 ...
Processing: 20240801 ...
Processing: 20240901 ...
Processing: 20241001 ...
Processing: 20241101 ...
Processing: 20241201 ...
Processing: 20250101 ...
Processing: 20250201 ...
Processing: 20250301 ...
Processing: 20250401 ...
Processing: 20250501 ...
Processing: 20250601 ...


In [ ]:
df_nyt = pd.DataFrame(all_nyt_data)
df_nyt

,headline,date,section,type,source,text_snippet,full_text
0,A Strongman President? These Voters Crave It.,2024-01-14,Opinion,Politics,New York Times,Authoritarian leaders project qualities that m...,Authoritarian leaders project qualities that m...
1,The Election No One Seems to Want Is Coming Ri...,2024-01-08,Opinion,Politics,New York Times,It’s 2024. Sorry.,It’s 2024. Sorry.
2,"Kamala Harris, Sharper and Lively, Begins to M...",2024-01-29,Opinion,Politics,New York Times,"Her skills are better, but can she revive the ...","Her skills are better, but can she revive the ..."
3,Transcript: Ezra Klein Interviews David French,2024-01-12,Opinion,Politics,New York Times,"The Jan. 12, 2023, episode of “The Ezra Klein ...","The Jan. 12, 2023, episode of “The Ezra Klein ..."
4,Biden Plans 2 Campaign Speeches to Underscore ...,2024-01-03,U.S.,Politics,New York Times,The president will speak at Valley Forge on th...,The president will speak at Valley Forge on th...
...,...,...,...,...,...,...,...
3303,Our New Podcast,2025-06-06,Briefing,Conflict,New York Times,"In “The Protocol,” we explore the controversia...","In “The Protocol,” we explore the controversia..."
3304,Israel Attacked Iran’s State TV,2025-06-16,Briefing,Conflict,New York Times,"Also, the suspect in the Minnesota killings co...","Also, the suspect in the Minnesota killings co..."
3305,"Starmer Picks Up Trump’s Papers, and 2 Small P...",2025-06-17,World,Conflict,New York Times,The British prime minister scrambled at Presid...,The British prime minister scrambled at Presid...
3306,A Primary and a Heat Wave,2025-06-25,Briefing,Conflict,New York Times,We’re covering the upset in New York City and ...,We’re covering the upset in New York City and ...


In [ ]:
nyt_filename = "nyt_2024_2025_full.csv"
df_nyt.to_csv(nyt_filename, index=False)
print(f"\nSaved to file: {nyt_filename}")


Saved to file: nyt_2024_2025_full.csv


In [ ]:
print("--- MERGING DATASETS ---")

# 1. Load the files we just created
try:
    df_g = pd.read_csv("guardian_2024_2025_full.csv")
    print(f"Loaded Guardian: {len(df_g)} rows")
except FileNotFoundError:
    print("Error: guardian_2024_full.csv not found. Run Cell 2 first.")
    df_g = pd.DataFrame()

try:
    df_n = pd.read_csv("nyt_2024_2025_full.csv")
    print(f"Loaded NYT: {len(df_n)} rows")
except FileNotFoundError:
    print("Error: nyt_2024_full.csv not found. Run Cell 3 first.")
    df_n = pd.DataFrame()

# 2. Combine
final_df = pd.concat([df_g, df_n], ignore_index=True)

# 3. Clean
# Drop duplicates (headlines often repeat in searches)
final_df.drop_duplicates(subset=['headline', 'date'], inplace=True)
# Drop rows with missing headlines
final_df.dropna(subset=['headline'], inplace=True)

# 4. Save Final Master File
output_filename = "final_10_dataset.csv"
final_df.to_csv(output_filename, index=False)

print("\n" + "="*40)
print(f"FINAL SUCCESS! DATASET READY.")
print(f"Final Count: {len(final_df)} rows")
print(f"Saved to: {output_filename}")
print("="*40)

# Check distribution
print("\nBreakdown by Source:")
print(final_df['source'].value_counts())
print("\nBreakdown by Topic:")
print(final_df['type'].value_counts())

--- MERGING DATASETS ---
Loaded Guardian: 7776 rows
Loaded NYT: 3308 rows

🎉 FINAL SUCCESS! DATASET READY.
Final Count: 10917 rows
Saved to: final_10_dataset.csv

Breakdown by Source:
source
The Guardian      7772
New York Times    3145
Name: count, dtype: int64

Breakdown by Topic:
type
Politics    4202
Conflict    3715
Economy     3000
Name: count, dtype: int64
